---
title: Query via Icechunk
---

# Query NISAR GUNW via Icechunk

Open the virtual Icechunk store created in the previous notebook and
query 10 random points. Only the chunks containing those points are
fetched from NASA's S3 storage.

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="Numcodecs codecs are not in the Zarr")
warnings.filterwarnings(
    "ignore", category=UserWarning, message=".*does not have a Zarr V3 specification.*"
)
warnings.filterwarnings("ignore", category=FutureWarning, module="earthaccess")

## Open the Icechunk store

In [ ]:
import time

import earthaccess
import icechunk
import xarray as xr
import zarr

zarr.config.set({"async.concurrency": 128})

earthaccess.login()

storage = icechunk.local_filesystem_storage("nisar-icechunk")
config = icechunk.RepositoryConfig.default()

# Configure access to the original data on NASA S3
bucket = "s3://sds-n-cumulus-prod-nisar-products"
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        bucket + "/",
        icechunk.s3_store(region="us-west-2"),
    ),
)

# Get temporary S3 credentials for reading virtual chunks
# Search for any NISAR GUNW granule to get the correct credential endpoint
results = earthaccess.search_data(
    short_name="NISAR_L2_GUNW_BETA_V1",
    point=(174.1192, -39.3379),
    temporal=("2026-01-01", "2026-01-04"),
)
s3_creds = earthaccess.get_s3_credentials(results=results)
virtual_credentials = icechunk.containers_credentials(
    {
        bucket + "/": icechunk.s3_credentials(
            access_key_id=s3_creds["accessKeyId"],
            secret_access_key=s3_creds["secretAccessKey"],
            session_token=s3_creds["sessionToken"],
        )
    }
)

repo = icechunk.Repository.open(
    storage=storage,
    config=config,
    authorize_virtual_chunk_access=virtual_credentials,
)

session = repo.readonly_session("main")
t0 = time.perf_counter()
ds = xr.open_zarr(session.store, consolidated=False, zarr_format=3)
print(f"Opened in {time.perf_counter() - t0:.2f}s")
ds

## Query 10 random points

In [ ]:
import numpy as np

varname = "unwrappedPhase"

rng = np.random.default_rng(42)
ny, nx = ds.dims["yCoordinates"], ds.dims["xCoordinates"]
yi = rng.integers(0, ny, size=10)
xi = rng.integers(0, nx, size=10)

t0 = time.perf_counter()
values = ds[varname].values[yi, xi]
elapsed = time.perf_counter() - t0

print(f"Queried 10 points in {elapsed:.2f}s")

## Results

In [ ]:
import pandas as pd

results = pd.DataFrame(
    {
        "y_index": yi,
        "x_index": xi,
        varname: values,
    }
)
results

## Visualize unwrapped phase

In [ ]:
import pygmt

fig = pygmt.Figure()
pygmt.makecpt(
    cmap="SCM/romaO", series=[0, 2 * np.pi, np.pi / 2], cyclic=True, continuous=True
)
fig.grdimage(grid=ds.unwrappedPhase, cmap=True, frame=True)
fig.colorbar()
fig.show()

## Visualize coherence magnitude

In [ ]:
fig = pygmt.Figure()
fig.grdimage(grid=ds.coherenceMagnitude, cmap="gray", frame=True)
fig.colorbar()
fig.show()